# temperature — 같은 질문, 다른 답

`1_1_openai` 에서는 `model` 과 `messages` 만 넣어 호출했습니다.
이 노트북은 여기에 **`temperature` 하나를 더 넣으면 무엇이 달라지는지** 직접 확인합니다.

## 이 노트북의 흐름

| 단계 | 내용 |
|---|---|
| **Step 1~2** | 준비 — 라이브러리, API 키, 클라이언트 |
| **Step 3** | 매번 같은 방식으로 묻는 작은 함수 만들기 |
| **Step 4~5** | 같은 질문을 3번씩 — `temperature=0` vs `temperature=1.5` |
| **Step 6** | 값을 조금씩 올려 가며 비교하기 |
| **Step 7** | 너무 높이면 생기는 일 |
| **Step 8** | 언제 얼마를 쓰나 |

> **이 노트북에서 기억할 한 가지**
> `temperature` 는 **답이 얼마나 들쭉날쭉해도 되는지**를 정하는 손잡이입니다.
> 낮으면 매번 비슷한 답, 높으면 매번 다른 답이 나옵니다.

## Step 1. 준비 — `1_1_openai` 와 똑같이

라이브러리를 가져오고, `.env` 의 키를 올리고, 클라이언트를 만듭니다.
`1_1_openai` 의 Step 1·2·4 를 한 셀에 모은 것입니다.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


## Step 2. temperature 가 무엇인가

LLM 은 답을 **한 토큰(단어 조각)씩** 만듭니다.
매번 "다음에 올 토큰 후보"마다 확률을 매기고, 그중 하나를 뽑습니다.

```
"오늘 날씨가 정말 ___"

  좋네요   60%
  덥네요   25%
  흐리네요 10%
  상쾌해요  5%
```

`temperature` 는 **이 확률표를 얼마나 고르게 펴 줄지**를 정합니다.

| temperature | 확률표가 어떻게 되나 | 결과 |
|---|---|---|
| `0` | 1등 후보에 거의 몰빵 | 거의 항상 "좋네요" — **매번 비슷한 답** |
| `1` (기본값) | 원래 확률 그대로 | 적당히 다양 |
| `1.5` 이상 | 꼴찌 후보도 자주 뽑힘 | "상쾌해요"도 자주 — **매번 다른 답** |

OpenAI 에서 쓸 수 있는 범위는 **`0` ~ `2`** 입니다.

> **주의 — 모델에 따라 못 쓰는 경우가 있습니다.**
> `1_1_openai` Step 5 에서 쓴 `gpt-5-nano` 같은 **추론(reasoning) 모델**은
> `temperature` 를 바꿀 수 없고, 넣으면 오류가 납니다.
> 그래서 이 노트북은 Step 9 에서 쓴 **`gpt-4.1-mini`** 를 씁니다.

## Step 3. 묻는 함수 하나 만들기

같은 질문을 temperature 만 바꿔 여러 번 보낼 것이므로, 호출 코드를 함수로 묶어 둡니다.

```python
def ask(prompt, temperature):
```

| 인자 | 하는 일 |
|---|---|
| `temperature` | 이번 노트북의 주인공 |
| `max_tokens=60` | 답을 짧게 끊는다. 여러 번 호출하므로 **요금과 시간을 아끼기 위해** |

`1_1_openai` 의 `create()` 호출에 **`temperature=` 한 줄이 늘었을 뿐**입니다.

In [2]:
MODEL = "gpt-4.1-mini"

def ask(prompt, temperature):
    response = openai.chat.completions.create(
        model = MODEL,
        messages = [{"role": "user", "content": prompt}],
        temperature = temperature,   # <- 새로 추가된 한 줄
        max_tokens = 60,             # 답을 짧게 (요금 절약)
    )
    return response.choices[0].message.content

## Step 4. `temperature=0` — 같은 질문을 3번

답이 여러 개 나올 수 있는 질문을 고릅니다.
"정답이 하나뿐인 질문(1+1은?)"은 temperature 를 바꿔도 차이가 안 보이기 때문입니다.

**결과를 보기 전에 예상해 보세요.** 세 답이 같을까요, 다를까요?

In [3]:
prompt = "새로 연 동네 카페 이름을 하나만 지어 주세요. 이름만 답하세요."

for i in range(3):
    print(f"{i+1}번째:", ask(prompt, temperature=0))

1번째: 모닝글로우 카페


2번째: 모닝글로우 카페


3번째: 모닝브리즈 카페


세 번 모두 **같거나 거의 같은 이름**이 나왔을 것입니다.
temperature 가 0 이면 모델이 매번 1등 후보만 고르기 때문입니다.

> **"완전히" 같지는 않을 수 있습니다.**
> 서버 내부 계산 사정으로 `0` 이어도 가끔 한두 글자가 달라집니다.
> "항상 똑같은 답"이 아니라 **"최대한 비슷한 답"** 으로 이해하세요.

## Step 5. `temperature=1.5` — 같은 질문을 다시 3번

질문은 그대로 두고 **temperature 만 1.5 로** 올립니다.

In [4]:
for i in range(3):
    print(f"{i+1}번째:", ask(prompt, temperature=1.5))

1번째: 아침영감


2번째: 온더테이블


3번째: 달빛정원


이번에는 **매번 다른 이름**이 나옵니다. 셀을 다시 실행하면 또 달라집니다.

| | Step 4 | Step 5 |
|---|---|---|
| `prompt` | 같음 | 같음 |
| `temperature` | `0` | `1.5` |
| 결과 | 거의 같은 답 반복 | 매번 다른 답 |

**코드에서 바꾼 것은 숫자 하나**입니다.

## Step 6. 값을 조금씩 올려 가며 비교

이번에는 조금 더 긴 글을 시켜서 **말투와 표현이 어떻게 변하는지** 봅니다.
`0` → `0.7` → `1.2` → `1.8` 순서로 올립니다.

In [5]:
prompt = "가을을 주제로 한 문장짜리 짧은 시를 써 주세요."

for t in [0, 0.7, 1.2, 1.8]:
    print(f"[temperature={t}]")
    print(ask(prompt, temperature=t))
    print()

[temperature=0]


낙엽 속에 스며드는 가을 햇살, 마음도 함께 물들다.

[temperature=0.7]


낙엽 속에 스며든 가을 햇살, 마음도 노랗게 물들다.

[temperature=1.2]


낙엽 춤추며 가을바람에 작은 속삭임이 번져간다.

[temperature=1.8]


깊어가는 가을, 낙엽들 속삭이며 추억을 땅에 남긴다.



보통 이런 경향이 보입니다.

| temperature | 경향 |
|---|---|
| `0` | 무난하고 예상 가능한 표현 ("낙엽", "단풍" 같은 단골 단어) |
| `0.7` | 자연스러우면서 약간의 변화 — **일상 대화에서 가장 많이 쓰는 구간** |
| `1.2` | 낯선 비유와 단어가 섞이기 시작 |
| `1.8` | 독특하지만 어색하거나 말이 안 되는 부분이 생김 |

## Step 7. 너무 높이면 — `temperature=2`

최댓값 `2` 로 올려 봅니다.

In [6]:
print(ask("인공지능을 한 문장으로 설명해 주세요.", temperature=2))

인공지능은 인간의 학습, 추론, 인식 등의 사고 과정을 컴퓨터 시스템으로 모방한 기술입니다.


문장이 엉뚱한 방향으로 흐르거나, 다른 언어·이상한 기호가 섞여 나오기도 합니다.
**매번 깨지는 것은 아니니** 셀을 몇 번 다시 실행해 보세요. 질문이 짧고 답이 짧을수록 멀쩡하게 나올 때가 많습니다.
꼴찌 후보 토큰까지 자주 뽑히다 보니 **한 번 어긋난 단어가 다음 단어를 더 어긋나게** 만들기 때문입니다.

> **다양함과 엉망은 한 끗 차이입니다.**
> "창의적인 답"이 필요해도 실무에서는 보통 `1.2` 를 넘기지 않습니다.

## Step 8. 언제 얼마를 쓰나

| 하고 싶은 일 | 추천 temperature | 이유 |
|---|---|---|
| 정보 추출, 분류, 요약 | `0` ~ `0.2` | 매번 같은 결과가 나와야 믿고 쓸 수 있다 |
| 코드 작성 | `0` ~ `0.3` | 엉뚱한 코드는 곧 버그 |
| 일반 대화, 질문 답변 | `0.5` ~ `0.8` | 자연스러우면서 크게 벗어나지 않는다 |
| 아이디어, 이름 짓기, 글쓰기 | `0.9` ~ `1.2` | 여러 번 뽑아서 고르는 용도 |

> 뒤에서 배울 **RAG** 는 문서에 있는 내용만 정확히 답해야 하므로
> 보통 `temperature=0` 으로 둡니다.

---

## 정리

```
create(model=..., messages=..., temperature=숫자)  ->  숫자가 클수록 답이 들쭉날쭉
```

### 꼭 기억할 것

| | 내용 |
|---|---|
| **temperature 는 다양성 손잡이** | 낮으면 비슷한 답, 높으면 다른 답 |
| **범위는 0 ~ 2** | 기본값은 1 |
| **0 이어도 완전히 같진 않다** | "최대한 비슷한 답"으로 이해 |
| **너무 높으면 엉망** | 실무에서는 보통 1.2 이하 |
| **추론 모델은 못 바꾼다** | `gpt-5-nano` 등에 넣으면 오류 |

### 직접 해 보기

1. Step 4 의 질문을 "1 더하기 1은?"으로 바꾸고 `temperature=1.5` 로 3번 물어 보세요. 왜 답이 안 바뀔까요?
2. `ask()` 에 `system` 메시지를 넣어 "당신은 시인입니다"라는 역할을 준 뒤 Step 6 을 다시 해 보세요.
3. `MODEL` 을 `"gpt-5-nano"` 로 바꾸고 실행해 보세요. 어떤 오류가 나나요?